In [8]:
import os
import time
import numpy as np
import pandas as pd
import joblib
import pygad
import pyswarms
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

In [9]:
best_models = ['dtree', 'logreg']

sizes = {'small': 50, 'large': 1500}
n_feats = {'small': 5, 'large': 12}

datasets = {}
for name, size in sizes.items():
    path = f"datasets/ds_size{size}_feat{n_feats[name]}.csv"
    df = pd.read_csv(path)
    datasets[name] = (df.drop('collision', axis=1), df['collision'])

param_ranges = {
    'dtree': {'max_depth': (1, 20), 'min_samples_split': (2, 20)},
    'logreg': {'C': (0.01, 100.0)} 
}

In [13]:
ga_results = []

for model_name in best_models:
    for data_name, (X, y) in datasets.items():
        if model_name == 'dtree':
            gene_space = [{'low': 2, 'high': 20, 'step': 1}, {'low': 2, 'high': 20, 'step': 1}]
            num_genes = 2
        else:
            gene_space = [{'low': 0.01, 'high': 100.0}]
            num_genes = 1
            
        def fitness_func(ga_instance, solution, solution_idx):
            try:
                if model_name == 'dtree':
                    max_d = int(solution[0])
                    min_split = int(solution[1])
                    max_d = None if max_d < 2 else max_d
                    model = DecisionTreeClassifier(max_depth=max_d, min_samples_split=min_split, random_state=42)
                else:
                    c_val = float(solution[0])
                    c_val = max(0.01, c_val)
                    model = LogisticRegression(C=c_val, max_iter=1000, solver='lbfgs', random_state=42)
                
                model.fit(X, y)
                # average='weighted' корректно работает при дисбалансе
                return f1_score(y, model.predict(X), average='weighted', zero_division=0)
            except:
                return 0.0

        ga_instance = pygad.GA(
            num_generations=30,
            num_parents_mating=4,
            fitness_func=fitness_func,
            sol_per_pop=15,
            num_genes=num_genes,
            gene_space=gene_space,
            mutation_num_genes=1,
            mutation_percent_genes=20,
            stop_criteria=None  # Убрали преждевременную остановку
        )
        
        t0 = time.perf_counter()
        ga_instance.run()
        elapsed = time.perf_counter() - t0
        
        best_sol, best_fit, _ = ga_instance.best_solution()
        ga_results.append({
            'model': model_name, 'dataset': data_name, 'method': 'GA',
            'best_params': best_sol, 'f1': best_fit, 'time': elapsed
        })
        print(f"GA {model_name} {data_name}: F1={best_fit:.4f}, Time={elapsed:.3f}s, Params={best_sol}")

ga_df = pd.DataFrame(ga_results)

GA dtree small: F1=0.0000, Time=0.648s, Params=[15. 17.]
GA dtree large: F1=0.0000, Time=0.632s, Params=[15. 15.]
GA logreg small: F1=0.0000, Time=0.404s, Params=[57.6062453]
GA logreg large: F1=0.0000, Time=0.542s, Params=[57.81043101]


In [11]:
def pso_objective(params, model_name, X, y):
    n_particles = params.shape[0]
    scores = np.zeros(n_particles)
    for i in range(n_particles):
        sol = params[i]
        if model_name == 'dtree':
            model = DecisionTreeClassifier(max_depth=int(sol[0]), min_samples_split=int(sol[1]), random_state=42)
        else:
            model = LogisticRegression(C=sol[0], max_iter=1000, random_state=42)
        try:
            model.fit(X, y)
            scores[i] = f1_score(y, model.predict(X))
        except:
            scores[i] = 0.0
    return -scores

pso_results = []

for model_name in best_models:
    for data_name, (X, y) in datasets.items():
        if model_name == 'dtree':
            bounds = (np.array([1, 2]), np.array([20, 20]))
        else:
            bounds = (np.array([0.01]), np.array([100.0]))
            
        options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
        optimizer = pyswarms.single.GlobalBestPSO(
            n_particles=10, dimensions=len(bounds[0]), options=options, bounds=bounds
        )
        
        t0 = time.perf_counter()
        cost, _ = optimizer.optimize(lambda p: pso_objective(p, model_name, X, y), iters=20, verbose=False)
        elapsed = time.perf_counter() - t0
        
        pso_results.append({
            'model': model_name, 'dataset': data_name, 'method': 'PSO',
            'f1': -cost, 'time': elapsed
        })
        print(f"PSO {model_name} {data_name}: F1={-cost:.4f}, Time={elapsed:.3f}s")

pso_df = pd.DataFrame(pso_results)

PSO dtree small: F1=0.0000, Time=0.203s
PSO dtree large: F1=0.0000, Time=0.283s
PSO logreg small: F1=0.0000, Time=0.163s
PSO logreg large: F1=0.0000, Time=0.211s


In [12]:
times_grid = {
    ('dtree', 'small'): 0.05, ('dtree', 'large'): 0.88,
    ('logreg', 'small'): 0.06, ('logreg', 'large'): 0.86
}

final_df = pd.concat([ga_df, pso_df])
final_df['grid_time'] = final_df.apply(lambda r: times_grid.get((r['model'], r['dataset']), 0), axis=1)
final_df['speedup'] = final_df['grid_time'] / final_df['time']

print("\n📊 Сравнение времени (Speedup > 1 значит, что GA/PSO быстрее GridSearch):")
print(final_df[['model', 'dataset', 'method', 'f1', 'time', 'grid_time', 'speedup']].to_string(index=False))

best_row = final_df.loc[final_df['f1'].idxmax()]
print(f"\n🏆 Лучшая модель: {best_row['model']} ({best_row['method']}) на {best_row['dataset']} данных с F1={best_row['f1']:.4f}")

X_full, y_full = datasets['large']
if best_row['model'] == 'dtree':
    model = DecisionTreeClassifier(max_depth=int(best_row['best_params'][0]) if 'best_params' in best_row else 5, random_state=42)
else:
    model = LogisticRegression(C=best_row['best_params'][0] if 'best_params' in best_row else 1.0, max_iter=1000, random_state=42)

model.fit(X_full, y_full)
os.makedirs('models_lab3', exist_ok=True)
joblib.dump(model, f"models_lab3/best_{best_row['model']}_{best_row['method']}.pkl")
print(f"✓ Модель сохранена: models_lab3/best_{best_row['model']}_{best_row['method']}.pkl")


📊 Сравнение времени (Speedup > 1 значит, что GA/PSO быстрее GridSearch):
 model dataset method  f1     time  grid_time   speedup
 dtree   small     GA 0.0 0.057915       0.05  0.863339
 dtree   large     GA 0.0 0.072825       0.88 12.083762
logreg   small     GA 0.0 0.045263       0.06  1.325580
logreg   large     GA 0.0 0.061053       0.86 14.086122
 dtree   small    PSO 0.0 0.202515       0.05  0.246895
 dtree   large    PSO 0.0 0.283400       0.88  3.105150
logreg   small    PSO 0.0 0.162576       0.06  0.369058
logreg   large    PSO 0.0 0.211123       0.86  4.073461


TypeError: unsupported format string passed to Series.__format__